In [ ]:
from pathlib import Path
import os, json, hashlib, platform, importlib.metadata
import numpy as np
import pandas as pd
import joblib
from sklearn.base import BaseEstimator, ClassifierMixin
from mapie.classification import SplitConformalClassifier

ROOT = Path(os.environ.get('PINPAD_DATA_ROOT', '/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate'))
OUTPUT = Path(os.environ.get('PINPAD_OUTPUT_ROOT', '/home/cse/pinpad/myenv/GNURadio_1/PINPAD_Compact_Results'))
CP_OUTPUT = Path(os.environ.get('PINPAD_CP_OUTPUT', str(OUTPUT / 'CP')))
CAPTURES_CSV = Path(os.environ.get('PINPAD_CP_CAPTURES', str(ROOT / 'cp_capture_manifest.csv')))
MODELS = ('DL', 'RF')
RUN_IDS = ('run1', 'run2', 'run3', 'run4', 'run5')
N_BUTTONS, N_PER_BUTTON, RESPONSE_LENGTH = (12, 200, 3624)
ALPHA = {'DL': 0.07, 'RF': 0.05}
TAU, MAX_CAPTURES = (1.0, 3)
SCORE = 'lac'
INCLUDE_LAST_LABEL = True
CP_SEED = 42
if importlib.metadata.version('MAPIE') != '1.0.1':
    raise RuntimeError('MAPIE 1.0.1 required')
TIE_RULE = 'lowest_class_index'
KEEP_PLACEMENT_BLOCKS_TOGETHER = True
RISK_LEVELS = np.arange(5, 31, dtype=float) / 100.0
THRESHOLDS = np.arange(0.5, 1.001, 0.1).round(2)
CAPTURE_BUDGETS = (1, 2, 3)
TEST = {'CC': [(c, 'R1') for c in range(16, 26)], 'CR': [(c, 'R3') for c in range(1, 6)], 'CCR': [(c, 'R2') for c in range(26, 36)]}
EXPECTED_COUNTS = {'CC': 24000, 'CR': 12000, 'CCR': 24000}
if SCORE not in ('aps', 'lac'):
    raise ValueError('SCORE must be aps or lac')

In [ ]:
def write_json(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + '\n')

def file_sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def array_hash(x):
    array = np.ascontiguousarray(x)
    return hashlib.sha256(array.view(np.uint8)).hexdigest()

def load_capture_manifest():
    f = pd.read_csv(CAPTURES_CSV, dtype=str, keep_default_na=False)
    required = {'domain', 'card', 'reader', 'button', 'sample_index', 'partition', 'episode_id', 'capture_index', 'session_id', 'capture_time'}
    if not required <= set(f):
        raise ValueError(f'Missing capture fields: {sorted(required - set(f))}')
    for col in ('card', 'button', 'sample_index'):
        f[col] = pd.to_numeric(f[col], errors='raise').astype(int)
    f['reader'] = f.reader.str.upper().str.strip()
    if set(f.domain) != set(TEST):
        raise ValueError('CC, CR, CCR required')
    if set(f.partition) != {'calibration', 'validation', 'final'}:
        raise ValueError('Invalid partition names')
    if not f.button.between(0, 11).all() or not f.sample_index.between(0, 199).all():
        raise ValueError('Invalid button/sample index')
    if f.duplicated(['card', 'reader', 'button', 'sample_index']).any():
        raise ValueError('Duplicate capture')
    for domain, g in f.groupby('domain'):
        if len(g) != EXPECTED_COUNTS[domain] or set(zip(g.card, g.reader)) != set(TEST[domain]):
            raise ValueError(f'{domain}: collection mismatch')
        if not g.groupby(['card', 'reader', 'button']).size().eq(200).all():
            raise ValueError('Expected 200 captures per button')
        for role, fraction in [('calibration', 0.1), ('validation', 0.1), ('final', 0.8)]:
            if len(g[g.partition == role]) != round(len(g) * fraction):
                raise ValueError(f'{domain}: {role} response count is not {fraction:.0%}')
            if set(g.loc[g.partition == role, 'button']) != set(range(12)):
                raise ValueError(f'{domain}/{role}: missing button classes')
        if KEEP_PLACEMENT_BLOCKS_TOGETHER:
            blocks = g.assign(block=g.sample_index // 50)
            if blocks.groupby(['card', 'reader', 'button', 'block']).partition.nunique().gt(1).any():
                raise ValueError('Placement block crosses partitions')
    linked = f[f.episode_id.ne('')]
    if linked.groupby(['domain', 'episode_id']).partition.nunique().gt(1).any():
        raise ValueError('Episode crosses partitions')
    episodes = f[f.partition != 'calibration']
    if episodes[['episode_id', 'session_id', 'capture_time', 'capture_index']].eq('').any().any():
        raise ValueError('Missing episode metadata')
    for key, g in episodes.groupby(['domain', 'episode_id']):
        if any((g[col].nunique() != 1 for col in ('partition', 'card', 'reader', 'button', 'session_id'))):
            raise ValueError(f'Mixed provenance/partition within episode {key}')
        g = g.assign(capture_index=pd.to_numeric(g.capture_index, errors='raise').astype(int)).sort_values('capture_index')
        if not 1 <= len(g) <= 3 or g.capture_index.tolist() != list(range(1, len(g) + 1)):
            raise ValueError(f'{key}: invalid capture order')
        times = pd.to_datetime(g.capture_time, utc=True, errors='raise')
        if times.isna().any() or (times.diff().iloc[1:] <= pd.Timedelta(0)).any():
            raise ValueError(f'{key}: timestamps must strictly increase')
    return f.reset_index(drop=True)

def load_test_responses(manifest):
    mapping_file = ROOT / 'collection_paths.csv'
    mapping = {}
    if mapping_file.is_file():
        table = pd.read_csv(mapping_file, dtype=str, keep_default_na=False)
        if not {'card', 'reader', 'directory'} <= set(table):
            raise ValueError('Invalid collection_paths.csv')
        for row in table.itertuples(index=False):
            key = (int(row.card), row.reader.strip().upper())
            if key in mapping or not row.directory.strip():
                raise ValueError('Duplicate/empty collection mapping')
            p = Path(row.directory).expanduser()
            mapping[key] = (p if p.is_absolute() else ROOT / p).resolve()
    expected = set(sum(TEST.values(), []))
    for card, reader in sorted(expected - set(mapping)):
        candidates = {p.resolve() for p in (ROOT / reader / f'card{card}', ROOT / reader.lower() / f'card{card}', ROOT / f'card{card}' / reader, ROOT / f'card{card}' / reader.lower()) if p.is_dir()}
        if len(candidates) != 1:
            raise FileNotFoundError(f'Provide unambiguous collection mapping: {card}/{reader}')
        mapping[card, reader] = candidates.pop()
    if len({mapping[k] for k in expected}) != len(expected):
        raise ValueError('Collections share a directory')
    responses = {}
    for domain in TEST:
        rows = manifest[manifest.domain == domain].copy().reset_index(drop=True)
        values = [None] * len(rows)
        for (card, reader, button), g in rows.groupby(['card', 'reader', 'button'], sort=False):
            folder = mapping[int(card), reader]
            hits = [p for p in (folder / f'button{button}.npy', folder / f'Button{button}.npy') if p.is_file()]
            if len(hits) != 1:
                raise FileNotFoundError(f'Expected one button{button}.npy in {folder}')
            raw = np.load(hits[0], allow_pickle=False)
            if raw.shape != (200, RESPONSE_LENGTH) or not np.iscomplexobj(raw) or (not np.isfinite(raw).all()):
                raise ValueError(f'Invalid response array: {hits[0]}')
            for index, row in g.iterrows():
                values[index] = raw[int(row.sample_index)]
        x = np.stack(values)
        rows['response_sha256'] = [array_hash(v) for v in x]
        if rows.groupby('response_sha256').partition.nunique().gt(1).any():
            raise ValueError(f'{domain}: waveform crosses partitions')
        measured = rows[rows.partition != 'calibration']
        if measured.response_sha256.duplicated().any():
            raise ValueError(f'{domain}: duplicate waveform')
        responses[domain] = (x, rows)
    combined = pd.concat([rows for x, rows in responses.values()], ignore_index=True)
    if combined.groupby('response_sha256').partition.nunique().gt(1).any():
        raise ValueError('Waveform crosses partitions')
    if combined[combined.partition != 'calibration'].response_sha256.duplicated().any():
        raise ValueError('Repeated evaluation waveform')
    return responses

In [ ]:
def preprocess(raw, prep):
    if prep['input_format'] != 'magnitude' or prep['normalization'] != 'source_global_zscore':
        raise ValueError('Expected magnitude preprocessing')
    x = np.asarray(raw)
    if x.ndim != 2 or x.shape[1] != prep['signal_length'] or (not np.iscomplexobj(x)) or (not np.isfinite(x).all()):
        raise ValueError('Invalid complex response shape or values')
    if prep['drift_mode'] == 'linear_complex':
        t = np.linspace(-1.0, 1.0, x.shape[1])
        x = x - x.mean(axis=1, keepdims=True)
        x = x - (x @ t / np.dot(t, t))[:, None] * t
    elif prep['drift_mode'] != 'none':
        raise ValueError('Unsupported saved drift mode')
    m = np.abs(x).astype(np.float32)[..., None]
    stats = prep['source_stats']
    mean = np.asarray(stats['mean'], dtype=np.float32)
    std = np.asarray(stats['std'], dtype=np.float32)
    if not np.isfinite(mean).all() or not np.isfinite(std).all() or np.any(std <= 0):
        raise ValueError('Invalid saved normalization')
    return ((m - mean) / std).astype(np.float32)

def load_predictor(name, run_id):
    directory = OUTPUT / name / run_id
    audit_path = directory / 'capture_hashes.csv'
    if not audit_path.is_file():
        raise FileNotFoundError(f'Missing training/development response audit: {audit_path}')
    hashes = set(pd.read_csv(audit_path, usecols=['response_sha256']).response_sha256.astype(str))
    if name == 'DL':
        import tensorflow as tf
        model_path = directory / 'adapted_deepcoral.keras'
        prep_path = directory / 'preprocessing.json'
        prep = json.loads(prep_path.read_text())
        model = tf.keras.models.load_model(model_path, compile=False)
        model.trainable = False

        def predict(raw):
            x = preprocess(raw, prep)
            if tuple(model.input_shape[1:]) != tuple(x.shape[1:]):
                raise ValueError('DL input shape mismatch')
            logits = np.asarray(model.predict(x, batch_size=256, verbose=0))
            if logits.shape != (len(x), 12):
                raise ValueError('Expected 12 linear logits')
            return checked_probabilities(tf.nn.softmax(logits, axis=1).numpy())
        metadata = dict(model_sha256=file_sha256(model_path), preprocessing_sha256=file_sha256(prep_path))
    elif name == 'RF':
        model_path = directory / 'RF_models.joblib'
        bundle = joblib.load(model_path)
        prep = bundle['preprocessing']
        pca, model = (bundle['pca'], bundle['rf'])
        if not np.array_equal(model.classes_, np.arange(12)):
            raise ValueError('RF class order is not B0–B11')

        def predict(raw):
            x = preprocess(raw, prep).reshape(len(raw), -1)
            return checked_probabilities(model.predict_proba(pca.transform(x)))
        metadata = dict(model_sha256=file_sha256(model_path), preprocessing=prep)
    else:
        raise ValueError('Unknown model')
    if prep['classes'] != [f'B{i}' for i in range(12)]:
        raise ValueError('Saved label order mismatch')
    return (predict, hashes, metadata)

def checked_probabilities(p):
    p = np.asarray(p, dtype=np.float64)
    if p.ndim != 2 or p.shape[1] != 12 or (not np.isfinite(p).all()) or np.any(p < 0) or np.any(p > 1):
        raise ValueError('Invalid class probabilities')
    if not np.allclose(p.sum(axis=1), 1.0, atol=1e-06):
        raise ValueError('Probabilities do not sum to one')
    return p / p.sum(axis=1, keepdims=True)

class FrozenProbabilities(ClassifierMixin, BaseEstimator):

    def __init__(self):
        self.classes_ = np.arange(12)
        self.n_features_in_ = 12

    def __sklearn_is_fitted__(self):
        return True

    def fit(self, X, y):
        raise RuntimeError('Model is frozen')

    def predict_proba(self, X):
        return checked_probabilities(X)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)

def calibrate(probabilities, labels, alpha):
    cp = SplitConformalClassifier(estimator=FrozenProbabilities(), confidence_level=1 - float(alpha), conformity_score=SCORE, prefit=True, random_state=CP_SEED)
    cp.conformalize(probabilities, labels)
    return cp

def get_sets(cp, probabilities):
    params = {'include_last_label': INCLUDE_LAST_LABEL} if SCORE == 'aps' else None
    _, sets = cp.predict_set(probabilities, conformity_score_params=params)
    sets = np.asarray(sets, dtype=bool)
    if sets.ndim == 3 and sets.shape[-1] == 1:
        sets = sets[:, :, 0]
    if sets.shape != (len(probabilities), 12):
        raise ValueError(f'Unexpected MAPIE set shape {sets.shape}')
    return sets

In [ ]:
def adaptive_decision(capture_sets, tau=1.0, budget=3):
    sets = np.asarray(capture_sets, dtype=bool)
    if sets.ndim != 2 or sets.shape[1] != 12 or (not len(sets)) or (not 1 <= budget <= 3) or (tau <= 0):
        raise ValueError('Invalid sets, threshold, or budget')
    scores = np.zeros(12, dtype=np.float64)
    for used, mask in enumerate(sets[:budget], 1):
        size = int(mask.sum())
        if size:
            scores[mask] += 1.0 / size
        accepted = bool(scores.max() >= tau)
        if accepted or used == budget:
            tied = np.flatnonzero(scores == scores.max())
            return dict(prediction=int(tied[0]), captures=used, reinputs=used - 1, accepted=accepted, forced=not accepted, tied=len(tied) > 1, scores=scores.tolist())
    raise ValueError('Episode ended before acceptance or budget')

def episode_layout(rows):
    cached = rows.attrs.get('episode_layout')
    if cached is not None:
        return cached
    names = []
    indices = []
    first = []
    for episode_id, g in rows.groupby('episode_id', sort=False):
        g = g.assign(capture_index=g.capture_index.astype(int)).sort_values('capture_index')
        names.append(episode_id)
        indices.append(g.index.to_numpy(dtype=int))
        first.append(int(g.index[0]))
    layout = np.full((len(names), 3), -1, dtype=int)
    lengths = np.asarray([len(ix) for ix in indices], dtype=int)
    for i, ix in enumerate(indices):
        layout[i, :len(ix)] = ix
    cached = (names, layout, lengths, rows.loc[first].copy())
    rows.attrs['episode_layout'] = cached
    return cached

def episode_metrics(rows, probabilities, sets, tau, budget, details=True):
    if rows.empty:
        raise ValueError('No episodes')
    names, layout, lengths, first = episode_layout(rows)
    n = len(names)
    scores = np.zeros((n, 12), dtype=np.float64)
    used = np.zeros(n, dtype=int)
    pred = np.full(n, -1, dtype=int)
    accepted = np.zeros(n, dtype=bool)
    tied = np.zeros(n, dtype=bool)
    active = np.ones(n, dtype=bool)
    for k in range(budget):
        if np.any(active & (lengths <= k)):
            raise ValueError('Episode ended before acceptance or budget')
        ix = np.flatnonzero(active)
        if not len(ix):
            break
        masks = sets[layout[ix, k]]
        sizes = masks.sum(axis=1)
        increments = np.divide(masks, sizes[:, None], out=np.zeros_like(masks, dtype=np.float64), where=sizes[:, None] > 0)
        scores[ix] += increments
        used[ix] = k + 1
        threshold = scores[ix].max(axis=1) >= tau
        finish = threshold | (k + 1 == budget)
        done = ix[finish]
        accepted[done] = threshold[finish]
        pred[done] = scores[done].argmax(axis=1)
        tied[done] = (scores[done] == scores[done].max(axis=1, keepdims=True)).sum(axis=1) > 1
        active[done] = False
    if active.any():
        raise RuntimeError('Incomplete adaptive decision')
    truth = first.button.to_numpy(dtype=int)
    initial = probabilities[layout[:, 0]].argmax(axis=1)
    processed = np.arange(3)[None, :] < used[:, None]
    capture_indices = layout[processed]
    capture_labels = rows.loc[capture_indices, 'button'].to_numpy(dtype=int)
    processed_sets = sets[capture_indices]
    set_sizes = processed_sets.sum(axis=1)
    covered = processed_sets[np.arange(len(capture_indices)), capture_labels]
    metrics = dict(episodes=int(n), processed_captures=int(len(capture_indices)), adaptive_accuracy=float(np.mean(pred == truth)), first_capture_accuracy=float(np.mean(initial == truth)), coverage=float(np.mean(covered)), mean_prediction_set_size=float(np.mean(set_sizes)), mean_reinputs=float(np.mean(used - 1)), mean_captures=float(np.mean(used)), acceptance_rate=float(np.mean(accepted)), forced_decision_rate=float(np.mean(~accepted)), tie_rate=float(np.mean(tied)), budget_reached_rate=float(np.mean(used == budget)))
    if not details:
        return (metrics, None, None)
    episodes = first.copy()
    episodes.attrs.clear()
    episodes['episode_id'] = names
    episodes['true_button'] = truth
    episodes['first_prediction'] = initial
    episodes['prediction'] = pred
    episodes['correct'] = pred == truth
    episodes['captures_used'] = used
    episodes['reinputs'] = used - 1
    episodes['accepted'] = accepted
    episodes['forced'] = ~accepted
    episodes['tied'] = tied
    episodes['evidence'] = [json.dumps(value.tolist()) for value in scores]
    episodes = episodes.reset_index(drop=True)
    captures = rows.loc[capture_indices].copy()
    captures.attrs.clear()
    captures['row_index'] = capture_indices
    captures['prediction_set_size'] = set_sizes
    captures['covered'] = covered
    captures['prediction_set'] = [json.dumps(np.flatnonzero(mask).tolist()) for mask in processed_sets]
    captures['model_prediction'] = probabilities[capture_indices].argmax(axis=1)
    captures['model_confidence'] = probabilities[capture_indices].max(axis=1)
    captures = captures.reset_index(drop=True)
    return (metrics, episodes, captures)

In [ ]:
def validation_report(name, run_id, predict, training_hashes, samples, run_output):
    records = []
    calibrators = {}
    cache = {}
    for domain, (raw, rows) in samples.items():
        if set(rows.response_sha256) & training_hashes:
            raise ValueError(f'{name}/{run_id}/{domain}: development overlap')
        cal = rows.index[rows.partition.eq('calibration')].to_numpy()
        val = rows.index[rows.partition.eq('validation')].to_numpy()
        p = np.full((len(rows), 12), np.nan, dtype=np.float64)
        p[cal] = predict(raw[cal])
        p[val] = predict(raw[val])
        labels = rows.button.to_numpy(dtype=int)
        validation_rows = rows.loc[val].copy()
        for alpha in RISK_LEVELS:
            cp = calibrate(p[cal], labels[cal], float(alpha))
            masks = np.zeros((len(rows), 12), dtype=bool)
            masks[val] = get_sets(cp, p[val])
            calibrators[domain, float(alpha)] = cp
            for tau in THRESHOLDS:
                for budget in CAPTURE_BUDGETS:
                    m, _, _ = episode_metrics(validation_rows, p, masks, float(tau), int(budget), details=False)
                    records.append(dict(model=name, run_id=run_id, domain=domain, alpha=float(alpha), tau=float(tau), max_captures=int(budget), **m))
        cache[domain] = p
        rows.to_csv(run_output / f'{domain}_capture_manifest.csv', index=False)
    grid = pd.DataFrame(records)
    grid.to_csv(run_output / 'validation_grid.csv', index=False)
    fixed = grid[np.isclose(grid.alpha, ALPHA[name]) & np.isclose(grid.tau, TAU) & grid.max_captures.eq(MAX_CAPTURES)]
    fixed.to_csv(run_output / 'validation_at_paper_point.csv', index=False)
    macro = grid.groupby(['alpha', 'tau', 'max_captures'], as_index=False)[['adaptive_accuracy', 'mean_reinputs', 'mean_prediction_set_size', 'coverage']].mean()
    chosen = macro.sort_values(['adaptive_accuracy', 'mean_reinputs', 'mean_prediction_set_size', 'alpha', 'tau', 'max_captures'], ascending=[False, True, True, True, True, True]).iloc[0]
    write_json(run_output / 'reference_validation_selection.json', chosen.to_dict())
    frozen = dict(model=name, run_id=run_id, alpha=ALPHA[name], tau=TAU, max_captures=MAX_CAPTURES, selection='manuscript operating point, fixed before final scoring', score=SCORE, include_last_label=INCLUDE_LAST_LABEL, tie_rule=TIE_RULE, cp_seed=CP_SEED)
    write_json(run_output / 'frozen_operating_point.json', frozen)
    return (calibrators, cache)

In [ ]:
def final_report(name, run_id, predict, samples, calibrators, cache, run_output):
    frozen = json.loads((run_output / 'frozen_operating_point.json').read_text())
    if frozen['alpha'] != ALPHA[name] or frozen['tau'] != TAU or frozen['max_captures'] != MAX_CAPTURES:
        raise ValueError('Operating point changed after validation')
    reports = []
    for domain, (raw, rows) in samples.items():
        final_ix = rows.index[rows.partition.eq('final')].to_numpy()
        p = cache[domain]
        p[final_ix] = predict(raw[final_ix])
        cp = calibrators[domain, float(ALPHA[name])]
        sets = np.zeros((len(rows), 12), dtype=bool)
        sets[final_ix] = get_sets(cp, p[final_ix])
        metrics, episodes, captures = episode_metrics(rows.loc[final_ix], p, sets, TAU, MAX_CAPTURES)
        reports.append(dict(model=name, run_id=run_id, domain=domain, alpha=ALPHA[name], tau=TAU, max_captures=MAX_CAPTURES, **metrics))
        episodes.to_csv(run_output / f'{domain}_episode_decisions.csv', index=False)
        captures.to_csv(run_output / f'{domain}_processed_captures.csv', index=False)
        np.savez_compressed(run_output / f'{domain}_final_predictions.npz', row_index=final_ix, probabilities=p[final_ix], prediction_sets=sets[final_ix], labels=rows.loc[final_ix, 'button'].to_numpy(int))
        joblib.dump(cp, run_output / f'{domain}_calibrator.joblib')
        scores = getattr(getattr(cp, '_mapie_classifier', None), 'conformity_scores_', None)
        if scores is not None:
            np.save(run_output / f'{domain}_calibration_scores.npy', np.asarray(scores))
    table = pd.DataFrame(reports)
    table.to_csv(run_output / 'final_domain_metrics.csv', index=False)
    return table

def summarize(all_metrics, destination):
    all_metrics.to_csv(destination / 'all_run_domain_metrics.csv', index=False)
    cols = ['adaptive_accuracy', 'first_capture_accuracy', 'coverage', 'mean_prediction_set_size', 'mean_reinputs']
    summary = all_metrics.groupby(['model', 'domain'])[cols].agg(['mean', 'std', 'count'])
    summary.columns = ['_'.join(v) for v in summary.columns]
    summary.to_csv(destination / 'across_run_summary.csv')
    macro = all_metrics.groupby(['model', 'run_id'], as_index=False)[cols].mean()
    macro.to_csv(destination / 'per_run_macro_metrics.csv', index=False)
    weighted = []
    for (name, run_id), g in all_metrics.groupby(['model', 'run_id']):
        row = dict(model=name, run_id=run_id)
        for key in ('adaptive_accuracy', 'first_capture_accuracy', 'mean_reinputs'):
            row[key] = float(np.average(g[key], weights=g.episodes))
        for key in ('coverage', 'mean_prediction_set_size'):
            row[key] = float(np.average(g[key], weights=g.processed_captures))
        weighted.append(row)
    pd.DataFrame(weighted).to_csv(destination / 'per_run_response_episode_weighted_metrics.csv', index=False)

In [ ]:
# def plot_risk_curves(destination):
#     import matplotlib.pyplot as plt
#     from matplotlib.ticker import FormatStrFormatter
#     plt.rcParams.update({'font.size': 8.5, 'pdf.fonttype': 42, 'ps.fonttype': 42})
#     colors = {'CC': '#D62728', 'CR': '#008000', 'CCR': '#003FBE'}
#     metrics = {'coverage': ('empirical_coverage', 'Empirical coverage (%)', 100), 'mean_prediction_set_size': ('average_prediction_set_size', 'Prediction set size', 1), 'adaptive_accuracy': ('adaptive_accuracy', 'Recognition accuracy (%)', 100), 'mean_reinputs': ('average_reinputs', 'Button re-inputs', 1)}
#     plot_dir = destination / 'validation_figures'
#     plot_dir.mkdir()
#     for name in MODELS:
#         grids = [pd.read_csv(destination / name / run_id / 'validation_grid.csv') for run_id in RUN_IDS]
#         grid = pd.concat(grids, ignore_index=True)
#         grid = grid[np.isclose(grid.tau, TAU) & grid.max_captures.eq(MAX_CAPTURES)]
#         for metric, (stem, ylabel, scale) in metrics.items():
#             summary = grid.groupby(['domain', 'alpha'])[metric].agg(['mean', 'std', 'count']).reset_index()
#             summary.to_csv(plot_dir / f'{name}_{stem}_validation_values.csv', index=False)
#             fig, ax = plt.subplots(figsize=(3.8, 2.8), layout='constrained')
#             for domain in TEST:
#                 g = summary[summary.domain.eq(domain)].sort_values('alpha')
#                 x = g.alpha.to_numpy()
#                 mean = g['mean'].to_numpy() * scale
#                 ax.plot(x, mean, color=colors[domain], linewidth=1.5, label=domain)
#                 if (g['count'] > 1).all():
#                     sd = g['std'].to_numpy() * scale
#                     ax.fill_between(x, mean - sd, mean + sd, color=colors[domain], alpha=0.18, linewidth=0)
#             ax.set(xlabel='Risk level ($\\alpha$)', ylabel=ylabel, xlim=(0.05, 0.3))
#             ax.set_xticks(np.arange(0.05, 0.301, 0.05))
#             ax.xaxis.set_major_formatter(FormatStrFormatter('%.2f'))
#             ax.tick_params(direction='in', top=True, right=True)
#             ax.grid(alpha=0.3, linestyle=':')
#             ax.legend(fontsize=8.5)
#             fig.savefig(plot_dir / f'{name}_{stem}_validation.pdf')
#             fig.savefig(plot_dir / f'{name}_{stem}_validation.png', dpi=600)
#             plt.close(fig)

In [ ]:
class AdaptivePINRecognizer:

    def __init__(self, name, run_id, domain, cp_output=CP_OUTPUT):
        directory = Path(cp_output) / name / run_id
        self.predict, _, model_reference = load_predictor(name, run_id)
        expected = json.loads((directory / 'model_reference.json').read_text())
        if model_reference != expected:
            raise ValueError('Checkpoint or preprocessing mismatch')
        self.settings = json.loads((directory / 'frozen_operating_point.json').read_text())
        if self.settings['score'] != SCORE or self.settings['include_last_label'] != INCLUDE_LAST_LABEL:
            raise ValueError('Score configuration mismatch')
        self.cp = joblib.load(directory / f'{domain}_calibrator.joblib')
        self.reset()

    def reset(self):
        self.evidence = np.zeros(12, dtype=np.float64)
        self.captures = 0

    def capture(self, complex_response):
        raw = np.asarray(complex_response)
        if raw.ndim != 1:
            raise ValueError('Expected one response')
        mask = get_sets(self.cp, self.predict(raw[None, :]))[0]
        size = int(mask.sum())
        if size:
            self.evidence[mask] += 1.0 / size
        self.captures += 1
        accepted = bool(self.evidence.max() >= self.settings['tau'])
        finished = accepted or self.captures >= self.settings['max_captures']
        result = dict(action='accept' if accepted else 'budget_fallback' if finished else 'request_reinput', prediction=int(self.evidence.argmax()) if finished else None, captures=self.captures, reinputs=self.captures - 1, prediction_set=np.flatnonzero(mask).tolist(), evidence=self.evidence.tolist(), accepted=accepted)
        if finished:
            self.reset()
        return result

In [ ]:
if CP_OUTPUT.exists():
    raise FileExistsError(f'Choose a new PINPAD_CP_OUTPUT directory: {CP_OUTPUT}')
manifest = load_capture_manifest()
samples = load_test_responses(manifest)
CP_OUTPUT.mkdir(parents=True)
versions = {p: importlib.metadata.version(p) for p in ('numpy', 'pandas', 'scikit-learn', 'MAPIE', 'joblib')}
write_json(CP_OUTPUT / 'experiment.json', dict(python=platform.python_version(), versions=versions, manifest_sha256=file_sha256(CAPTURES_CSV), models=list(MODELS), runs=list(RUN_IDS), score=SCORE, include_last_label=INCLUDE_LAST_LABEL, cp_seed=CP_SEED, tie_rule=TIE_RULE, placement_blocks_kept_together=KEEP_PLACEMENT_BLOCKS_TOGETHER, risk_levels=RISK_LEVELS.tolist(), risk_increment=0.01, risk_sweep_partition='validation', final_alpha=ALPHA, response_hash_convention='SHA256 of contiguous original-dtype waveform bytes; must match training capture_hashes.csv', metric_unit='accuracy/reinputs per observed episode; coverage/set size per processed capture', unspecified_manuscript_details=['score', 'APS inclusion convention', 'ties', 'original episode assignments', 'original CP split assignments', 'validation selection rule', 'seed']))
print('Risk levels:', RISK_LEVELS)
print('Risk sweep: validation, increment 0.01')
print('Final operating points:', ALPHA)
reports = []
for name in MODELS:
    for run_id in RUN_IDS:
        predict, development_hashes, model_metadata = load_predictor(name, run_id)
        run_output = CP_OUTPUT / name / run_id
        run_output.mkdir(parents=True)
        write_json(run_output / 'model_reference.json', model_metadata)
        calibrators, cache = validation_report(name, run_id, predict, development_hashes, samples, run_output)
        report = final_report(name, run_id, predict, samples, calibrators, cache, run_output)
        reports.append(report)
        print(report.to_string(index=False))
        del predict, calibrators, cache
all_metrics = pd.concat(reports, ignore_index=True)
summarize(all_metrics, CP_OUTPUT)
print('Saved CP reports:', CP_OUTPUT)
# plot_risk_curves(CP_OUTPUT)